In [1]:
!pip install -q h2o

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import h2o
from h2o.automl import H2OAutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score

In [3]:
from google.colab import files
uploaded=files.upload()

Saving GermanCredit.xlsx to GermanCredit.xlsx


In [4]:
df=pd.read_excel("GermanCredit.xlsx")

print(df.shape)
df.head()

(1000, 21)


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,... < 100 DM,6,critical account/other credits existing,domestic appliances,1169,unknown/no savings account,... >= 7 years,4,male : single,none,...,real estate,67,none,own,2,skilled employee/official,1,yes,yes,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,real estate,22,none,own,1,skilled employee/official,1,no,yes,0
2,no checking account,12,critical account/other credits existing,retraining,2096,... < 100 DM,4 <= ... < 7 years,2,male : single,none,...,real estate,49,none,own,1,unskilled - resident,2,no,yes,1
3,... < 100 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,building society savings agreement/life insurance,45,none,for free,1,skilled employee/official,2,no,yes,1
4,... < 100 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,none,...,unknown/no property,53,none,for free,2,skilled employee/official,2,no,yes,0


In [5]:
## Separate features and target
X=df.drop(columns=["credit_risk"])
y=df["credit_risk"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:",len(X_train))
print("Testing rows:",len(X_test))

Training rows: 800
Testing rows: 200


In [6]:
## Start H2O and prepare data
import h2o
from h2o.automl import H2OAutoML

h2o.init()
train_df=X_train.copy()
train_df["credit_risk"]=y_train.values

train_h2o=h2o.H2OFrame(train_df)
train_h2o["credit_risk"]=train_h2o["credit_risk"].asfactor()

test_h2o=h2o.H2OFrame(X_test.copy())

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: [0.050s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate; [0.050s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate; openjdk version "21.0.12.1" 2026-08-18; OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu); OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.13/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp71vkceah
  JVM stdout: /tmp/tmp71vkceah/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp71vkceah/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server a

H2O_cluster_uptime:,04 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 28 days
H2O_cluster_name:,H2O_from_python_unknownUser_q0iojx
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.152 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


h2o.init() starts the H2O engine.

copy() keeps the original data unchanged.

We add the target only to the training data.

H2OFrame() converts pandas data to H2O format.

asfactor() tells H2O that credit_risk is a classification target.

The test data intentionally excludes the target to prevent it from being used as an input feature.


In [8]:
## run AutoML
aml=H2OAutoML(
    max_models=10,
    seed=42,
    sort_metric="AUC"
)
aml.train(
    y="credit_risk",
    training_frame=train_h2o
)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/5
# GBM base models (used / total),0/1
# XGBoost base models (used / total),1/1
# GLM base models (used / total),1/1
# DRF base models (used / total),2/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


max_models=10 limits the number of candidate models.

seed=42 helps make results reproducible.

sort_metric="AUC" ranks models using AUC.

aml.train() trains and compares the candidate models automatically.


In [9]:
## View the leaderboard
leaderboard=aml.leaderboard
print(leaderboard)

model_id                                                     auc    logloss     aucpr    mean_per_class_error      rmse       mse
StackedEnsemble_BestOfFamily_1_AutoML_1_20261010_73611  0.794345   0.485365  0.897532                0.33869   0.401307  0.161048
StackedEnsemble_AllModels_1_AutoML_1_20261010_73611     0.79128    0.487675  0.896307                0.394643  0.402633  0.162114
GLM_1_AutoML_1_20261010_73611                           0.790993   0.489563  0.894953                0.341667  0.402303  0.161848
GBM_1_AutoML_1_20261010_73611                           0.774487   0.509001  0.887579                0.417262  0.411809  0.169587
GBM_2_AutoML_1_20261010_73611                           0.772891   0.507803  0.8837                  0.4       0.41078   0.16874
GBM_3_AutoML_1_20261010_73611                           0.770558   0.504811  0.877188                0.375298  0.408338  0.16674
XGBoost_2_AutoML_1_20261010_73611                       0.77006    0.523009  0.878894       

In [11]:
  ## evaluate best model
predictions=aml.leader.predict(test_h2o)
pred_df=predictions.as_data_frame()

best_pred=pred_df["predict"].astype(int)
best_prob=pred_df["p1"]

print("Accuracy:",accuracy_score(y_test,best_pred))
print(classification_report(y_test,best_pred))
print("ROC-AUC:",roc_auc_score(y_test,best_prob))

stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
Accuracy: 0.745


/usr/local/lib/python3.13/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


              precision    recall  f1-score   support

           0       0.62      0.40      0.48        60
           1       0.78      0.89      0.83       140

    accuracy                           0.74       200
   macro avg       0.70      0.65      0.66       200
weighted avg       0.73      0.74      0.73       200

ROC-AUC: 0.7867857142857142
